##### 1. Check table: **cepalplatform_SmeBpsmLbsPaymentSchedule**

In [0]:
%sql
/*
-- Check 1: Number of rows in the table = 238899 = COUNT(DISTINCT ID)
-- Check 2: Όλες οι εγγραφές στο παρακάτω query επιστρέφουν 1 με εξαίρεση 2 LBSParentIDS =  296 & 1199 με InstallmentSN = 1 (δες το query πιο κάτω)
%sql
SELECT LBSParentID, InstallmentSN, COUNT(*) AS CountRows
FROM `d-cdh`.landed.cepalplatform_SmeBpsmLbsPaymentSchedule
WHERE IsActive IS TRUE
GROUP BY LBSParentID, InstallmentSN
ORDER BY CountRows DESC
LIMIT 10

%sql
SELECT *
FROM `d-cdh`.landed.cepalplatform_SmeBpsmLbsPaymentSchedule
WHERE LBSParentID IN (296, 1199)
AND InstallmentSN = 1
ORDER BY LBSParentID DESC

-- Check 3:
-- Keys of the table: **LBSParentID, InstallmentSN, InstallmentDate (or CreationDate)**
SELECT LBSParentID, InstallmentSN, InstallmentDate, COUNT(*) AS CountRows
FROM `d-cdh`.landed.cepalplatform_SmeBpsmLbsPaymentSchedule
WHERE IsActive IS TRUE
GROUP BY LBSParentID, InstallmentSN, InstallmentDate
ORDER BY CountRows DESC
LIMIT 10

-- Check 4:
-- Εάν βγάλουμε το **WHERE IsActive IS TRUE** τότε λαμβάνουμε τρυάδες με πολλαπλά rows, ex.
-- LBSParentID = 1347 & InstallmentSN = 2 & InstallmentDate = '2021-12-24' με CountRows = 31 (δες το query below)
-- Όλες οι εγγραφες αυτού όμως του παραδείματος έχουν IsActive = False, με εξαίρεση μία
SELECT LBSParentID, InstallmentSN, InstallmentDate, COUNT(*) AS CountRows
FROM `d-cdh`.landed.cepalplatform_SmeBpsmLbsPaymentSchedule
GROUP BY LBSParentID, InstallmentSN, InstallmentDate
ORDER BY CountRows DESC
LIMIT 10

SELECT *
FROM `d-cdh`.landed.cepalplatform_SmeBpsmLbsPaymentSchedule
WHERE LBSParentID = 1347
AND InstallmentSN = 2
AND InstallmentDate = '2021-12-24'
ORDER BY IsActive
*/

-- Check 5: Keys of the table: **LBSParentID, InstallmentSN, InstallmentDate (or CreationDate)**
SELECT LBSParentID, InstallmentSN, CreationDate, COUNT(*) AS countRows
FROM `d-cdh`.landed.cepalplatform_SmeBpsmLbsPaymentSchedule
GROUP BY  LBSParentID, InstallmentSN, CreationDate
ORDER BY countRows DESC
LIMIT 2

#### Create Stored Procedures

In [0]:
import os, sys
import logging
import datetime
from datetime import datetime
from py4j.java_gateway import JavaObject  
from pyspark.sql import SparkSession, DataFrame
from typing import Optional
from pyspark.sql.window import Window
from pyspark.sql.functions import (
    coalesce,
    concat_ws,
    current_timestamp,
    expr,
    from_utc_timestamp,
    lit,
    max,
    sum,
    col,
    sha2,
    substring,
    to_date,
    when,
    broadcast,
    trim,
    row_number
)

def load_landed_delta_table(
    spark: SparkSession,
    catalog_landed: str,
    table_name: str,
    table_filters: str,
    table_selected_fields: list[str],
) -> DataFrame:
    
    df = (
        spark.read.table(f"{catalog_landed}.{table_name}")
        .filter(table_filters)
        .select(table_selected_fields)
    )

    # Trim all string columns
    for col_name, col_type in df.dtypes:
        if col_type == "string":
            df = df.withColumn(col_name, trim(df[col_name]))

    return df

##### 1. Stored procedure that populates the table: **sme_business_plan_lbspaymentschedule**

In [0]:
cluster_columns = ["BusinessDate"]
run_date = '2026-02-18'
date_filter = f"businessdate = {run_date!r}"
catalog_landed = "`d-cdh`.landed"

# ============================== cepalplatform_smebpsmlbspaymentschedule ====================================

lbs_payment_schedule_extra_filters = None
if lbs_payment_schedule_extra_filters is not None:
    lbs_payment_schedule_extra_filters.append(date_filter)
    lbs_payment_schedule_final_filters = " AND ".join(
        lbs_payment_schedule_extra_filters
    )
else:
    lbs_payment_schedule_final_filters = date_filter

lbs_payment_schedule_selected_filters = [
    "LBSParentID",
    "InstallmentSN",
    "InstallmentDate",
    "InstallmentTotalAmount",
    "InstallmentCapitalAmount",
    "InstallmentChargeAmount",
    "InstallmentN128Amount",
    "InstallmentTotalChargeAmount",
    "InstallmentRemainingCapitalAmount",
    "InstallmentNPVAmount",
    "InstallmentNPVCalculationDate",
    "CreationDate",
    "CreationUser",
    "IsActive",
    "BusinessDate",
    ]

lbs_payment_schedule = load_landed_delta_table(
    spark,
    catalog_landed,  
    "cepalplatform_smebpsmlbspaymentschedule",
    lbs_payment_schedule_final_filters,
    lbs_payment_schedule_selected_filters,
    )

# Keys of the table: **LBSParentID, InstallmentSN, InstallmentDate (or CreationDate)**
lbs_payment_schedule = lbs_payment_schedule.filter(col("IsActive") == True)

# =============================== Target table =================================
target_lbs_payment_schedule = lbs_payment_schedule.select(
    lbs_payment_schedule.LBSParentID.cast("Int"),
    lbs_payment_schedule.InstallmentSN.cast("Int"),
    lbs_payment_schedule.InstallmentDate.cast("Timestamp"),
    lbs_payment_schedule.InstallmentTotalAmount.cast("Decimal(18, 4)"),
    lbs_payment_schedule.InstallmentCapitalAmount.cast("Decimal(18, 4)"),
    lbs_payment_schedule.InstallmentChargeAmount.cast("Decimal(18, 4)"),
    lbs_payment_schedule.InstallmentN128Amount.cast("Decimal(18, 4)"),
    lbs_payment_schedule.InstallmentTotalChargeAmount.cast("Decimal(18, 4)"),
    lbs_payment_schedule.InstallmentRemainingCapitalAmount.cast("Decimal(18, 4)"),
    lbs_payment_schedule.InstallmentNPVAmount.cast("Decimal(18, 4)"),
    lbs_payment_schedule.InstallmentNPVCalculationDate.cast("Timestamp"),
    lbs_payment_schedule.CreationDate.cast("Timestamp"),
    lbs_payment_schedule.CreationUser.cast("String"),
    lbs_payment_schedule.IsActive.cast("Boolean"),
    lbs_payment_schedule.BusinessDate.cast("Date"),
)


##### 2. Stored procedure that populates the table: **sme_business_plan_contract**

In [0]:
cluster_columns = ["BusinessDate"]
run_date = '2026-02-18'
date_filter = f"businessdate = {run_date!r}"
catalog_landed = "`d-cdh`.landed"

# ============================== cepalplatform_smebusinessplancontract ====================================

business_plan_contract_extra_filters = None
if business_plan_contract_extra_filters is not None:
    business_plan_contract_extra_filters.append(date_filter)
    business_plan_contract_final_filters = " AND ".join(
        business_plan_contract_extra_filters
    )
else:
    business_plan_contract_final_filters = date_filter

business_plan_contract_selected_filters = [
    "ID",
    "AgreementNumber",
    "BusinessPlanStrategyManagementID",
    "creationdate",
    "CurrencyCode",
    "BusinessDate",
    ]

business_plan_contract = load_landed_delta_table(
    spark,
    catalog_landed,  
    "cepalplatform_smebusinessplancontract",
    business_plan_contract_final_filters,
    business_plan_contract_selected_filters,
    )

# ============================== cepalplatform_smebusinessplancontractloan ====================================

business_plan_contract_loan_extra_filters = None
if business_plan_contract_loan_extra_filters is not None:
    business_plan_contract_loan_extra_filters.append(date_filter)
    business_plan_contract_loan_final_filters = " AND ".join(
        business_plan_contract_loan_extra_filters
    )
else:
    business_plan_contract_loan_final_filters = date_filter

business_plan_contract_loan_selected_filters = [
    "AccountNumber",
    "LoanStatus",
    "BusinessPlanContractID"
    ]

business_plan_contract_loan = load_landed_delta_table(
    spark,
    catalog_landed,  
    "cepalplatform_smebusinessplancontractloan",
    business_plan_contract_loan_final_filters,
    business_plan_contract_loan_selected_filters,
    )

# ============================== cepalplatform_smebusinessplancontractcollateral ====================================

business_plan_contract_collateral_extra_filters = None
if business_plan_contract_collateral_extra_filters is not None:
    business_plan_contract_collateral_extra_filters.append(date_filter)
    business_plan_contract_collateral_final_filters = " AND ".join(
        business_plan_contract_collateral_extra_filters
    )
else:
    business_plan_contract_collateral_final_filters = date_filter

business_plan_contract_collateral_selected_filters = [
    "ID",
    "BusinessPlanContractID",
    "ABCollateralID",
    "ThesisCollateralID",
    "OwnershipID",
    "ABPropertyKey",
    "ThesisPropertyKey",
    ]

business_plan_contract_collateral = load_landed_delta_table(
    spark,
    catalog_landed,  
    "cepalplatform_smebusinessplancontractcollateral",
    business_plan_contract_collateral_final_filters,
    business_plan_contract_collateral_selected_filters,
    )

# ============================== joins ====================================

business_plan_contract_loan_collateral = (
    business_plan_contract
    .join(
        business_plan_contract_loan,
        business_plan_contract.ID == business_plan_contract_loan.BusinessPlanContractID,
        "left"
    )
    .join(
        business_plan_contract_collateral,
        business_plan_contract.ID == business_plan_contract_collateral.BusinessPlanContractID,
        "left"
    )                       
)

# ============================== business_contract schema ====================================

target_business_plan_contract_loan_collateral = business_plan_contract_loan_collateral.select(
    business_plan_contract.ID.cast("Int"),
    business_plan_contract.BusinessPlanStrategyManagementID.cast("Int").alias("BusinessPlanStrategy_ID"),
    business_plan_contract.AgreementNumber.cast("String").alias("Agreement_Number"),
	business_plan_contract.creationdate.cast("Date").alias("CreationDate"),
	business_plan_contract_loan.AccountNumber.alias("Account_Number"),
	business_plan_contract_collateral.ABCollateralID.cast("String"),
	business_plan_contract_collateral.ThesisCollateralID.cast("String"),
	business_plan_contract_collateral.OwnershipID.cast("String"),
	business_plan_contract_collateral.ABPropertyKey.cast("String"),
	business_plan_contract_collateral.ThesisPropertyKey.cast("String"),
    business_plan_contract.CurrencyCode.cast("String"),
    business_plan_contract_loan.LoanStatus.cast("String"),
    business_plan_contract.BusinessDate.cast("Date"),
)

# ΠΡΟΣΟΧΗ: Respective check should be performed because: Count rows of the table **target_business_contract** = 196,243 is different with the 
# Count rows of the target table **`p-cdh`.landed.sme_business_plan_contract BusinessDate = '2026-02-03'** = 196,472
# SOLUTION: The test has been passed by populating the 3 laning/landed table with data for the BuisnessDate = '2026-02-18' and re-running the experiment.ISSUE SOLVED BY TESTING

##### 3. Stored procedure that populates the table: **sme_business_plan**

In [0]:
# dev-Landed:DataMode:cepalplatform_SmeBusinessPlan | cepalplatform_SmeBusinessPlan
# dev-Landed:DataMode:cepalplatform_SmeRfdtBpStatus | cepalplatform_SmeRfdtBpStatus
# dev-Landed:DataMode:cepalplatform_SmeRfdtBpSolutionType | cepalplatform_SmeRfdtBpSolutionType
cluster_columns = ["BusinessDate"]
run_date = '2026-02-18'
date_filter = f"businessdate = {run_date!r}"
catalog_landed = "`d-cdh`.landed"

# ============================== cepalplatform_smebusinessplan ====================================

business_plan_extra_filters = None
if business_plan_extra_filters is not None:
    business_plan_extra_filters.append(date_filter)
    business_plan_final_filters = " AND ".join(
        business_plan_extra_filters
    )
else:
    business_plan_final_filters = date_filter

business_plan_selected_filters = [
    "ID",
    "StatusID",
    "SolutionTypeID",
    "DebtorCDI",
    "DebtorAFM",
    "DebtorBankGroupID",
    "IntentionDate",
    "CompletionTargetDate",
    "ApplicationDate",
    "Comments",
    "CreationDate",
    "CreationUser",
    "LastModifiedDate",
    "LastModifiedUser",
    "PortfolioID",
    "BusinessDate",
    ]

business_plan = load_landed_delta_table(
    spark,
    catalog_landed,  
    "cepalplatform_smebusinessplan",
    business_plan_final_filters,
    business_plan_selected_filters,
    )

# ============================== cepalplatform_smerfdtbpstatus ====================================

business_plan_status_extra_filters = None
if business_plan_status_extra_filters is not None:
    business_plan_status_extra_filters.append(date_filter)
    business_plan_status_final_filters = " AND ".join(
        business_plan_status_extra_filters
    )
else:
    business_plan_status_final_filters = date_filter

business_plan_status_selected_filters = ["ID", "Label"]

business_plan_status = load_landed_delta_table(
    spark,
    catalog_landed,  
    "cepalplatform_smerfdtbpstatus",
    business_plan_status_final_filters,
    business_plan_status_selected_filters,
    )

# ============================== cepalplatform_smerfdtbpsolutiontype ====================================

business_plan_solution_type_extra_filters = None
if business_plan_solution_type_extra_filters is not None:
    business_plan_solution_type_extra_filters.append(date_filter)
    business_plan_solution_type_final_filters = " AND ".join(
        business_plan_solution_type_extra_filters
    )
else:
    business_plan_solution_type_final_filters = date_filter

business_plan_solution_type_selected_filters = ["ID", "Label"]

business_plan_solution_type = load_landed_delta_table(
    spark,
    catalog_landed,  
    "cepalplatform_smerfdtbpsolutiontype",
    business_plan_solution_type_final_filters,
    business_plan_solution_type_selected_filters,
    )

# ============================== joins =======================================

business_plan_status_solution_type = (
    business_plan
    .join(
        business_plan_status,
        business_plan.StatusID == business_plan_status.ID,
        "left"
    )
    .join(
        business_plan_solution_type,
        business_plan.SolutionTypeID == business_plan_solution_type.ID,
        "left"
    )
)

# ============================== target table =======================================

target_business_plan_status_solution_type = business_plan_status_solution_type.select(
    business_plan.ID.cast("Int"),
    business_plan.DebtorCDI.cast("String"),
    business_plan.DebtorAFM.cast("String"),
    business_plan.DebtorBankGroupID.cast("String").alias("GroupCode"),
	business_plan_status.Label.cast("String").alias("Business_plan_status"),
	business_plan_solution_type.Label.cast("String").alias("SolutionType"), 
    business_plan.IntentionDate.cast("Date"), 
    business_plan.CompletionTargetDate.cast("Date"),
    business_plan.ApplicationDate.cast("Date"),
    business_plan.Comments.cast("String"),
    business_plan.CreationDate.cast("Timestamp"),
    business_plan.CreationUser.cast("String"),
    business_plan.LastModifiedDate.cast("Timestamp"),
    business_plan.LastModifiedUser.cast("String"),
    business_plan.PortfolioID.cast("Int"),
    business_plan.BusinessDate.cast("Date"),
)

# IMPORTANT: To be checked why the count of the table **target_business_plan_status_solution_type** does not match 
#            with output of the following query: 
#            SELECT COUNT(*) AS countRows 
#            FROM  `p-cdh`.landed.sme_business_plan WHERE BusinessDate = '2026-02-04'
# Due to BusinessDate data incosistency, the test has been passed by bringing data into the 3 tables: cepalplatform_SmeBusinessPlan, cepalplatform_SmeRfdtBpStatus 
# cepalplatform_SmeRfdtBpSolutionType and conducting the experiment (again) for BusinessDate = '2026-02-18'. ISSUE SOLVED BY TESTING

##### 4. Stored procedure that populates the table: **sme_business_plan_strategy**

In [0]:
# dev-Landed:DataMode:cepalplatform_SmeBusinessPlan
# dev-Landed:DataMode:cepalplatform_SmeRfdtBpStatus 
# dev-Landed:DataMode:cepalplatform_SmeRfdtBpSolutionType
 
cluster_columns = ["BusinessDate"]
run_date = '2026-02-05'
date_filter = f"businessdate = {run_date!r}"
catalog_landed = "`d-cdh`.landed"

# ============================== cepalplatform_smebusinessplanstrategymanagement ====================================

business_plan_strategy_management_extra_filters = None
if business_plan_strategy_management_extra_filters is not None:
    business_plan_strategy_management_extra_filters.append(date_filter)
    business_plan_strategy_management_final_filters = " AND ".join(
        business_plan_strategy_management_extra_filters
    )
else:
    business_plan_strategy_management_final_filters = date_filter

business_plan_strategy_management_selected_filters = [
    "ID",
    "StrategyID",
    "StatusID",
    "BusinessPlanID",
    "DraftedByID",
    "ProposedByID",
    "ConfidenceID",
    "BeginMonth",
    "Comments",
    "CreationDate",
    "CreationUser",
    "LastModifiedDate",
    "LastModifiedUser",
    "BusinessDate",
    ]

business_plan_strategy_management = load_landed_delta_table(
    spark,
    catalog_landed,  
    "cepalplatform_smebusinessplanstrategymanagement",
    business_plan_strategy_management_final_filters,
    business_plan_strategy_management_selected_filters,
    )

# ============================== cepalplatform_smerfdtbpstrategymanagementstatus ====================================

business_plan_strategy_management_status_extra_filters = None
if business_plan_strategy_management_status_extra_filters is not None:
    business_plan_strategy_management_status_extra_filters.append(date_filter)
    business_plan_strategy_management_status_final_filters = " AND ".join(
        business_plan_strategy_management_status_extra_filters
    )
else:
    business_plan_strategy_management_status_final_filters = date_filter

business_plan_strategy_management_status_selected_filters = [
    "ID",
    "Label",
    "Priority",
    "BusinessDate",
    ]

business_plan_strategy_management_status = load_landed_delta_table(
    spark,
    catalog_landed,  
    "cepalplatform_smerfdtbpstrategymanagementstatus",
    business_plan_strategy_management_status_final_filters,
    business_plan_strategy_management_status_selected_filters,
    )

# ============================== cepalplatform_smerfdtbpdraftedby ====================================

business_plan_drafted_by_extra_filters = None
if business_plan_drafted_by_extra_filters is not None:
    business_plan_drafted_by_extra_filters.append(date_filter)
    business_plan_drafted_by_final_filters = " AND ".join(
        business_plan_drafted_by_extra_filters
    )
else:
    business_plan_drafted_by_final_filters = date_filter

business_plan_drafted_by_selected_filters = ["ID", "Label", "BusinessDate"]

business_plan_drafted_by = load_landed_delta_table(
    spark,
    catalog_landed,  
    "cepalplatform_smerfdtbpdraftedby",
    business_plan_drafted_by_final_filters,
    business_plan_drafted_by_selected_filters,
    )

# ============================== cepalplatform_smerfdtbpproposedby ====================================

business_plan_proposed_by_extra_filters = None
if business_plan_proposed_by_extra_filters is not None:
    business_plan_proposed_by_extra_filters.append(date_filter)
    business_plan_proposed_by_final_filters = " AND ".join(
        business_plan_proposed_by_extra_filters
    )
else:
    business_plan_proposed_by_final_filters = date_filter

business_plan_proposed_by_selected_filters = ["ID", "Label", "BusinessDate"]

business_plan_proposed_by = load_landed_delta_table(
    spark,
    catalog_landed,  
    "cepalplatform_smerfdtbpproposedby",
    business_plan_proposed_by_final_filters,
    business_plan_proposed_by_selected_filters,
    )

# ============================== cepalplatform_smerfdtbpconfidence ====================================

business_plan_confidence_extra_filters = None
if business_plan_confidence_extra_filters is not None:
    business_plan_confidence_extra_filters.append(date_filter)
    business_plan_confidence_final_filters = " AND ".join(
        business_plan_confidence_extra_filters
    )
else:
    business_plan_confidence_final_filters = date_filter

business_plan_confidence_selected_filters = ["ID", "Label", "BusinessDate"]

business_plan_confidence = load_landed_delta_table(
    spark,
    catalog_landed,  
    "cepalplatform_smerfdtbpconfidence",
    business_plan_confidence_final_filters,
    business_plan_confidence_selected_filters,
    )

# ============================== cepalplatform_smerfdtbpstrategy ====================================

business_plan_strategy_extra_filters = None
if business_plan_strategy_extra_filters is not None:
    business_plan_strategy_extra_filters.append(date_filter)
    business_plan_strategy_final_filters = " AND ".join(
        business_plan_strategy_extra_filters
    )
else:
    business_plan_strategy_final_filters = date_filter

business_plan_strategy_selected_filters = ["ID", "Label", "BusinessDate"]

business_plan_strategy = load_landed_delta_table(
    spark,
    catalog_landed,  
    "cepalplatform_smerfdtbpstrategy",
    business_plan_strategy_final_filters,
    business_plan_strategy_selected_filters,
    )

# ============================== joins ====================================

business_plan_strategy_management_joins = (
    business_plan_strategy_management
    .join(
        business_plan_strategy_management_status,
        business_plan_strategy_management_status.ID == business_plan_strategy_management.StatusID,
        "left"
        )
    .join(
        business_plan_drafted_by,
        business_plan_drafted_by.ID == business_plan_strategy_management.DraftedByID,
        "left"
        )
    .join(
        business_plan_strategy,
        business_plan_strategy.ID == business_plan_strategy_management.StrategyID,
        "left"
        )
    .join(
        business_plan_proposed_by,
        business_plan_proposed_by.ID == business_plan_strategy_management.ProposedByID,
        "left"
        )
    .join(
        business_plan_confidence,
        business_plan_confidence.ID == business_plan_strategy_management.ConfidenceID,
        "left"
    )

)

target_business_plan_strategy_management = business_plan_strategy_management_joins.select(
    business_plan_strategy_management.ID.cast("Int"),
    business_plan_strategy_management.BusinessPlanID.cast("Int"),
    business_plan_strategy_management_status.Label.cast("String").alias("BPSStatus"),
    business_plan_strategy_management_status.Priority.cast("Int").alias("BPSStatusPriority"),
    business_plan_drafted_by.Label.cast("String").alias("DraftBy"),
    business_plan_strategy.Label.cast("String").alias("Strategy"),
    business_plan_strategy_management.BeginMonth.cast("Date"),
    business_plan_proposed_by.Label.cast("String").alias("ProposedBy"),
    business_plan_confidence.Label.cast("String").alias("Confidence"),
    business_plan_strategy_management.Comments.cast("String"),
    business_plan_strategy_management.CreationDate.cast("Timestamp"),
    business_plan_strategy_management.CreationUser.cast("String"),
    business_plan_strategy_management.LastModifiedDate.cast("Timestamp"),
    business_plan_strategy_management.LastModifiedUser.cast("String"),
    business_plan_strategy_management.StrategyID.cast("Int"),
    business_plan_strategy_management.BusinessDate.cast("Date"), 
)

target_business_plan_strategy_management.createOrReplaceTempView("target_business_plan_strategy_management")